In [ ]:
#This version is a full variable load version

In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# t_res='5min'; res='1km'
# res='1km'
# Np_str='1e6'

# # dx = 1 km; Np = 1M; Nt = 1 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6_5min.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6_5min.nc') #***
# t_res='5min'; res='1km'
# Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# # dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***
# res='250m'
# Np_str='150e6'

In [2]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [ ]:
###########################################################################################################################################################################

In [ ]:
#LOADING VARIABLES
###############################################################

In [3]:
# Loading Important Variables
##############
if 'emptylike' not in globals():
    print('loading neccessary variables')
    variable='lfc'; LFC_data=data[variable].data #get w data
    variable='lcl'; LCL_data=data[variable].data #get w data
    print('done')
    empty_like=True 

loading neccessary variables
done


In [4]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
out_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'
with h5py.File(out_file, 'r') as f:
    parcel_z = f['z'][:]
    
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [ ]:
#MAKING LAGRANGIAN BINARY ARRAY
###############################################################

In [5]:
LFC=np.zeros_like(Z,dtype='bool')
LCL=np.zeros_like(Z,dtype='bool')

Nt=len(data['time'])
Np=len(parcel['xh'])
for p in np.arange(Np):
    if np.mod(p,25e3)==0: print(f"{p}/{len(parcel['xh'])}")

    #Get Indicies
    zs=parcel_z[:,p]
    ys=Y[:,p]
    xs=X[:,p]
    ts = np.arange(Nt)  

    #Get Values
    lfcs = LFC_data[ts, ys, xs]
    lcls = LCL_data[ts, ys, xs]
    
    LFC[:,p]=(zs>=lfcs)#*1
    LCL[:,p]=(zs>=lcls)#*1

0/1000000
25000/1000000
50000/1000000
75000/1000000
100000/1000000
125000/1000000
150000/1000000
175000/1000000
200000/1000000
225000/1000000
250000/1000000
275000/1000000
300000/1000000
325000/1000000
350000/1000000
375000/1000000
400000/1000000
425000/1000000
450000/1000000
475000/1000000
500000/1000000
525000/1000000
550000/1000000
575000/1000000
600000/1000000
625000/1000000
650000/1000000
675000/1000000
700000/1000000
725000/1000000
750000/1000000
775000/1000000
800000/1000000
825000/1000000
850000/1000000
875000/1000000
900000/1000000
925000/1000000
950000/1000000
975000/1000000


In [6]:
# Saving Data
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
out_file=dir2+f'LFC_LCL_binary_array_{res}_{t_res}_{Np_str}.h5'
with h5py.File(out_file, 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('LFC', data=LFC, dtype='bool') #binary array for general updraft (w>=0.1)
    f.create_dataset('LCL', data=LCL, dtype='bool') #binary array for general updraft (w>=0.5 & qc+qi>=1e-6)

In [ ]:
#########################################

In [21]:
# # Reading Back Data Later
# ##############
# import h5py
# dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# with h5py.File(dir2+f'LFC_LCL_binary_array.h5', 'r') as f:
#     # Load the dataset by its name
#     LFC = f['LFC'][:]
#     LCL = f['LCL'][:]